# Mini Project 2: Monthly Expense Analyser

**The scenario:** The finance team has sent you five CSV files — one per department — containing January and February expense claims. Your job is to:

1. Load all files automatically (without naming each one manually)
2. Clean and validate the data, handling errors gracefully
3. Merge everything into one dataset
4. Analyse spending by department and category
5. Flag any suspiciously large expenses
6. Export a clean summary report

**What's new in this project compared to Project 1:**
- You will write **functions** — reusable blocks of code you define once and call many times
- You will handle **errors** properly, so the script doesn't crash on bad data
- You will use **pandas** — Python's most important library for data analysis
- You will use **sets** to find unique values across datasets
- You will use **os** — a built-in library for working with files and folders

**Important:** In this project, some cells ask you to complete the code yourself.  
Look for the `# YOUR CODE HERE` comments.

---

## Step 1 — Find all the files automatically

In Project 1 we opened one file by name. Here we have five. In real life you might have fifty.  
Instead of writing `open('expenses_sales.csv')`, `open('expenses_marketing.csv')` ... we let Python find them.

**New concept: the `os` library**  
`os` gives Python access to the operating system — file names, folder paths, environment variables.  
Think of it as Python's file explorer.

**New concept: list comprehension**  
A compact way to build a list from a loop, all on one line:
```python
# Long way:
result = []
for x in my_list:
    if x > 0:
        result.append(x)

# List comprehension — same thing:
result = [x for x in my_list if x > 0]
```

In [ ]:
import os

# os.listdir('.') returns a list of every file in the current folder
all_files = os.listdir('.')
print('All files in folder:', all_files)
print()

# We only want files that start with 'expenses_' and end with '.csv'
# List comprehension filters them in one line
expense_files = [
    f for f in all_files
    if f.startswith('expenses_') and f.endswith('.csv')
]

expense_files.sort()   # alphabetical order — predictable, easier to debug
print(f'Found {len(expense_files)} expense files:')
for f in expense_files:
    print(' -', f)

**Why does this matter?**  
If next month six departments submit files, this code still works without any changes.  
Hard-coding file names is brittle; discovering them automatically is robust.

---

## Step 2 — Write a cleaning function

In Project 1 we wrote the cleaning logic inline inside the loop. That was fine for one file.  
Now we have five files — we don't want to copy-paste the same cleaning code five times.

**New concept: functions**  
A function is a named, reusable block of code. You define it once with `def`, then call it by name whenever you need it.

```python
def greet(name):          # 'def' defines the function, 'name' is the input (parameter)
    message = 'Hello, ' + name
    return message        # 'return' sends a result back to whoever called the function

greet('Laura')            # → 'Hello, Laura'
greet('Marco')            # → 'Hello, Marco'
```

**New concept: error handling with `try / except`**  
Some operations can fail — like converting a string to a number.  
`try / except` lets you attempt something risky and handle failure gracefully instead of crashing.

```python
try:
    result = float('€€567.00')   # this will fail
except ValueError:
    result = None                # instead of crashing, we store None
```

In [ ]:
def clean_amount(raw_text):
    """
    Converts a raw amount string like '€342.00' or '€€567.00' into a float.
    Returns None if the conversion fails.
    
    The triple-quoted string at the start of a function is called a 'docstring'.
    It documents what the function does — always a good habit.
    """
    # Remove ALL euro signs (handles the double-€ bug in some files)
    cleaned = raw_text.replace('€', '').strip()
    
    if not cleaned:         # empty string after cleaning
        return None
    
    try:
        return float(cleaned)
    except ValueError:
        # ValueError is raised when float() can't parse the string
        # Instead of crashing the whole script, we return None
        return None


# Test the function on several cases before using it on real data
test_cases = ['€342.00', '€€567.00', '  €89.99  ', '', 'broken']

print('Testing clean_amount():')
for raw in test_cases:
    result = clean_amount(raw)
    print(f"  Input: {repr(raw):20s}  →  Output: {result}")

**Notice:** We tested the function in isolation before plugging it into the main script.  
This is standard professional practice — test small pieces, then combine them.

Now write a second function yourself:

In [ ]:
def clean_date(raw_date):
    """
    Cleans a date string by stripping whitespace.
    Returns None if the date is empty.
    """
    # YOUR CODE HERE
    # Hint: use .strip() to remove whitespace, then check if the result is empty
    # Return the cleaned string if valid, or None if empty
    pass   # delete this line when you add your code


# Test your function:
print(clean_date('2024-01-05'))   # should print: 2024-01-05
print(clean_date('  '))           # should print: None
print(clean_date(''))             # should print: None

---
## Step 3 — Load and merge all files

Now we loop through the file list, load each one, clean each row using our functions,  
and collect everything into a single list.

**New concept: `sets`**  
A set is like a list, but it only keeps **unique** values — no duplicates.  
It's perfect for answering "what are all the distinct categories across all files?"

```python
categories = set()           # empty set
categories.add('Travel')     # {'Travel'}
categories.add('Meals')      # {'Travel', 'Meals'}
categories.add('Travel')     # {'Travel', 'Meals'}  ← no duplicate added
```

In [ ]:
import csv

all_rows = []         # will hold every clean row from every file
all_categories = set()   # unique spending categories across all departments
skipped = []          # track rows we couldn't use, with reasons

for filename in expense_files:
    department_rows = 0
    
    try:
        # The file open itself can fail — e.g. file is locked or corrupted
        with open(filename, 'r', encoding='utf-8-sig') as f:
            reader = csv.DictReader(f)
            
            for row in reader:
                # Clean the critical fields using our functions
                date   = clean_date(row.get('date', ''))
                amount = clean_amount(row.get('amount', ''))
                dept   = row.get('department', '').strip()
                cat    = row.get('category', '').strip()
                desc   = row.get('description', '').strip()
                appr   = row.get('approved_by', '').strip()
                
                # Validate: skip rows missing critical data
                if not date:
                    skipped.append({'file': filename, 'reason': 'missing date', 'row': dict(row)})
                    continue
                if amount is None:
                    skipped.append({'file': filename, 'reason': f'bad amount: {row["amount"]}', 'row': dict(row)})
                    continue
                if not dept or not cat:
                    skipped.append({'file': filename, 'reason': 'missing dept/category', 'row': dict(row)})
                    continue
                
                # Build the clean row
                clean_row = {
                    'date':        date,
                    'department':  dept,
                    'category':    cat,
                    'description': desc,
                    'amount':      amount,
                    'approved_by': appr
                }
                
                all_rows.append(clean_row)
                all_categories.add(cat)   # sets ignore duplicates automatically
                department_rows += 1
                
    except FileNotFoundError:
        print(f'WARNING: {filename} not found — skipping.')
    except Exception as e:
        print(f'ERROR reading {filename}: {e}')
    else:
        # 'else' on a try/except runs only if NO exception occurred
        print(f'{filename}: loaded {department_rows} rows')

print(f'\nTotal clean rows: {len(all_rows)}')
print(f'Rows skipped:     {len(skipped)}')
print(f'\nUnique categories found (as a set):')
print(all_categories)

In [ ]:
# Inspect what was skipped and why
if skipped:
    print(f'{len(skipped)} rows were skipped:\n')
    for s in skipped:
        print(f"  File: {s['file']}")
        print(f"  Reason: {s['reason']}")
        print(f"  Row: {s['row']}")
        print()

**What the `try / except / else` structure gives us:**  
- The script doesn't crash when it hits a bad amount like `€€567.00`  
- We know exactly which rows were skipped and why  
- We can fix the source files and re-run with confidence

This is how production-grade scripts behave: they fail gracefully and report problems.

---

## Step 4 — Analyse with pandas

So far we've used only Python's built-in tools. Now we bring in **pandas** — the most widely used  
library for data analysis in Python. It's what Excel would be if Excel were programmable.

**New concept: libraries**  
A library (also called a package or module) is code written by someone else that you can use in your project.  
Python has thousands of them. `pandas` is one of the most important for business use.

**New concept: DataFrame**  
A pandas DataFrame is a table — rows and columns, just like a spreadsheet.  
Once your data is in a DataFrame, pandas gives you powerful tools: filtering, grouping, sorting, aggregating.

```python
import pandas as pd        # 'as pd' gives it a short alias — convention in the industry

df = pd.DataFrame(list_of_dicts)   # build a table from our list
df.head()                          # show the first 5 rows
df['amount'].sum()                 # sum the entire amount column
```

In [ ]:
import pandas as pd

# Convert our list of dicts into a pandas DataFrame
df = pd.DataFrame(all_rows)

# Convert 'date' column to a proper date type so pandas understands it
df['date'] = pd.to_datetime(df['date'], dayfirst=True, errors='coerce')

print('Shape of the dataset:', df.shape)   # (rows, columns)
print()
print('Column types:')
print(df.dtypes)
print()
print('First 5 rows:')
df.head()

In [ ]:
# --- Summary statistics ---
print('=== Overall spend ===')
print(f"Total expenses:   €{df['amount'].sum():,.2f}")
print(f"Average expense:  €{df['amount'].mean():,.2f}")
print(f"Largest expense:  €{df['amount'].max():,.2f}")
print(f"Smallest expense: €{df['amount'].min():,.2f}")
print()

# --- Spend by department (groupby = the pandas version of a pivot table) ---
print('=== Spend by department ===')
dept_summary = df.groupby('department')['amount'].agg(
    total='sum',
    count='count',
    average='mean'
).round(2).sort_values('total', ascending=False)

print(dept_summary)

In [ ]:
# --- Spend by category across all departments ---
print('=== Spend by category ===')
cat_summary = df.groupby('category')['amount'].sum().sort_values(ascending=False).round(2)
print(cat_summary)
print()

# --- Your turn: spend by department AND category ---
print('=== Spend by department and category ===')
# YOUR CODE HERE
# Hint: groupby() can accept a list of columns, e.g. groupby(['col1', 'col2'])
# Try to show total spend for each department-category combination


---
## Step 5 — Flag anomalies

Finance teams often need to flag unusually large expenses for review.  
We'll define "suspicious" as anything more than 2 standard deviations above the mean —  
a standard statistical approach for outlier detection.

**Pandas makes this easy:** column operations work on the whole column at once,  
just like an Excel formula applied to an entire column.

In [ ]:
# Calculate the threshold: mean + 2 standard deviations
mean_amount   = df['amount'].mean()
std_amount    = df['amount'].std()
threshold     = mean_amount + (2 * std_amount)

print(f'Mean expense:      €{mean_amount:,.2f}')
print(f'Std deviation:     €{std_amount:,.2f}')
print(f'Anomaly threshold: €{threshold:,.2f}')
print()

# Boolean filtering — this is one of pandas' most powerful features
# df['amount'] > threshold  returns a True/False for every row
# df[condition] returns only the rows where condition is True
anomalies = df[df['amount'] > threshold].copy()
anomalies = anomalies.sort_values('amount', ascending=False)

print(f'Flagged expenses ({len(anomalies)} rows):')
print(anomalies[['date', 'department', 'category', 'description', 'amount', 'approved_by']].to_string(index=False))

In [ ]:
# Add a flag column to the main dataset
df['flagged'] = df['amount'] > threshold

flagged_count = df['flagged'].sum()    # True counts as 1, False as 0
flagged_total = df[df['flagged']]['amount'].sum()

print(f'{flagged_count} expenses flagged for review')
print(f'Total value of flagged expenses: €{flagged_total:,.2f}')
print(f'That is {flagged_total / df["amount"].sum() * 100:.1f}% of total spend')

---
## Step 6 — Write a report function and export

Now we package the export logic into a function — so it's reusable and testable.  
We'll export two files: the full cleaned dataset, and the department summary.

**Functions with multiple parameters:**  
Functions can take more than one input. Parameters can also have **default values**,  
which means the caller doesn't have to provide them every time.

```python
def save_report(df, filename, include_flagged=True):   # include_flagged has a default
    ...

save_report(df, 'report.csv')              # include_flagged defaults to True
save_report(df, 'report.csv', False)       # override the default
```

In [ ]:
def export_to_csv(dataframe, filename, index=False):
    """
    Saves a pandas DataFrame to a CSV file.
    Returns True if successful, False if something goes wrong.
    """
    try:
        dataframe.to_csv(filename, index=index, encoding='utf-8')
        row_count = len(dataframe)
        print(f'Saved: {filename}  ({row_count} rows)')
        return True
    except PermissionError:
        # This happens if the file is open in Excel
        print(f'ERROR: Could not save {filename} — is it open in Excel?')
        return False
    except Exception as e:
        print(f'ERROR saving {filename}: {e}')
        return False


# --- Export 1: Full cleaned dataset ---
export_to_csv(df, 'expenses_all_clean.csv')

# --- Export 2: Department summary ---
dept_summary_export = dept_summary.reset_index()   # turns the index into a regular column
export_to_csv(dept_summary_export, 'expenses_summary_by_dept.csv')

# --- Export 3: Flagged anomalies only ---
export_to_csv(anomalies, 'expenses_flagged.csv')

In [ ]:
# YOUR TURN: write a function that exports the category summary
# It should:
#   1. Accept the category summary Series and a filename as parameters
#   2. Reset the index so 'category' becomes a column
#   3. Rename the column 'amount' to 'total_spend'
#   4. Call export_to_csv() to save it

def export_category_summary(cat_series, filename):
    """
    Exports the category summary to CSV.
    """
    # YOUR CODE HERE
    pass


# Call your function:
export_category_summary(cat_summary, 'expenses_summary_by_category.csv')

---
## Step 7 — Reflect and extend

You've built a multi-file data pipeline with error handling, reusable functions, and pandas analysis.

**Discussion questions:**
1. In Step 3, two rows were skipped because of `€€` double-symbol bugs. How would you find and fix the source files?
2. Our anomaly detection uses a fixed statistical rule. Can you think of a case where a large expense is legitimate and should NOT be flagged?
3. Where in this script would you add a function to send an email alert when anomalies are found?

**Extension challenge A — monthly breakdown:**  
The DataFrame has a `date` column. Use `df['date'].dt.month` to group expenses by month  
and show whether spending increased from January to February.

In [ ]:
# Extension A: monthly breakdown
# YOUR CODE HERE
# Hint: df['month'] = df['date'].dt.month  creates a new column
# Then groupby('month') works just like groupby('department')


**Extension challenge B — top spender per department:**  
For each department, find the single most expensive claim.  
Hint: look up `df.groupby('department')['amount'].idxmax()`.

In [ ]:
# Extension B: top expense per department
# YOUR CODE HERE


---
## Concepts covered across both projects

| Concept | Project 1 | Project 2 |
|---|---|---|
| Variables, data types | reading amounts, regions | amounts, thresholds, counters |
| Strings | `.lower()`, `.replace()`, `.strip()` | `.startswith()`, `.endswith()`, `.strip()` |
| Lists | `clean_rows`, `append()` | `all_rows`, `skipped`, `expense_files` |
| Dictionaries | region totals | per-row data, skipped log |
| Sets | — | `all_categories` — unique values |
| Control structures | `if/else`, `for`, `continue` | `if/else`, `for`, `continue` |
| Functions | — | `clean_amount()`, `clean_date()`, `export_to_csv()` |
| Error handling | — | `try/except/else`, `ValueError`, `PermissionError` |
| File handling | `open()`, `csv.reader`, `csv.writer` | `os.listdir()`, multiple files, `pd.to_csv()` |
| Libraries | `csv` (built-in) | `os`, `csv`, `pandas` |
| Pandas | — | DataFrame, `groupby()`, `agg()`, filtering, `to_csv()` |